# Lezione 6. Controllo senza modello: SARSA e Q-learning

**Durata stimata:** 105 minuti
**Capitolo del manuale:** [07_control_sin_modelo.md](../manual/07_control_sin_modelo.md)

## Dal valutare al migliorare

Nella lezione precedente hai imparato a valutare una politica fissa: dato un agente che agisce sempre allo stesso modo, hai calcolato quanto si aspetta di guadagnare da ogni stato, con Monte Carlo (MC) e con la Differenza Temporale (TD, dall'inglese *Temporal Difference*). Questa è la **predizione**, e lascia una domanda aperta: e se la politica di partenza non è buona? Misurare il valore di una strategia mediocre non la rende migliore.

Oggi fai il passo successivo: il **controllo**. Non ti accontenti più di misurare quanto vale una politica, vuoi che l'agente la migliori mentre impara, episodio dopo episodio. Conoscerai due algoritmi centrali dell'apprendimento per rinforzo (RL, dall'inglese *Reinforcement Learning*) moderno: SARSA (da *State-Action-Reward-State-Action*) e Q-learning. Entrambi imparano senza modello —senza conoscere in anticipo come funziona l'ambiente— e partono dalla stessa domanda ("quanto è buona questa azione in questo stato?"), ma rispondono in modo diverso, con conseguenze pratiche molto differenti.

### Obiettivi di oggi

- Spiegare l'Iterazione di Politica Generalizzata (GPI) e perché il controllo richiede Q(s,a) invece di V(s)
- Distinguere l'aggiornamento on-policy di SARSA dall'aggiornamento off-policy di Q-learning
- Implementare l'ambiente Cliff Walking e addestrare entrambi gli algoritmi su di esso
- Confrontare la ricompensa durante l'addestramento e i percorsi finali appresi da ciascuno

> 🎙️ **Traccia della lezione:** apri chiedendo "se sapete già misurare quanto vale una strategia, come la migliorereste?". Raccogli un paio di risposte prima di introdurre la GPI: l'idea di alternare valutazione e miglioramento di solito emerge da sola.


## 1. Dal prevedere al migliorare: l'Iterazione di Politica Generalizzata

Immagina di imparare a cucinare un piatto nuovo. Lo prepari, lo assaggi, correggi un po' la ricetta, lo rifai, lo riassaggi e correggi di nuovo. Non aspetti di padroneggiare la ricetta perfetta sulla teoria prima di cucinare per la prima volta: alterni tra "valutare com'è venuto" e "migliorare la ricetta", più e più volte, finché ti avvicini a qualcosa di eccellente.

Questa alternanza è l'idea alla base dell'**Iterazione di Politica Generalizzata** (GPI, dall'inglese *Generalized Policy Iteration*), un principio che struttura quasi tutto il controllo nell'RL:

1. **Valutazione**: stima quanto vale ogni azione sotto la politica attuale
2. **Miglioramento**: aggiusta la politica in modo da favorire le azioni con il valore stimato migliore
3. **Ripeti**: torna al passo uno con la politica migliorata

Non serve completare ogni passo alla perfezione perché funzioni: puoi valutare un po', migliorare un po', valutare ancora un po', e quella versione "a metà" converge comunque verso la politica ottimale nel caso tabulare (quando memorizzi un valore esplicito per ogni combinazione di stato e azione). SARSA e Q-learning sono due modi diversi di eseguire la GPI passo passo, aggiornando il valore dopo ogni singola azione invece di aspettare la fine dell'episodio.

Una differenza importante rispetto alla lezione precedente: nel controllo non basta più stimare V(s). Poiché non hai un modello dell'ambiente, ti serve il valore di ogni **azione** in ogni stato, Q(s,a): così puoi migliorare la politica scegliendo l'azione con Q più alto, senza simulare cosa accadrebbe con ciascuna alternativa.


## 2. Il dilemma dell'esplorazione: ε-greedy e GLIE

Se l'agente sceglie sempre l'azione che oggi sembra migliore, non scopre mai se un'altra azione, non ancora provata, sia in realtà superiore. Per questo il controllo ha bisogno di esplorare. La strategia più comune è la politica **ε-greedy** (epsilon-greedy): con probabilità ε l'agente sceglie un'azione a caso (esplora), e con probabilità 1−ε sceglie l'azione con il valore Q più alto noto finora (sfrutta).

Per garantire che questo processo converga davvero verso la politica ottimale esiste una condizione formale chiamata **GLIE** (dall'inglese *Greedy in the Limit with Infinite Exploration*, "golosa nel limite con esplorazione infinita"). Richiede due cose: che ogni combinazione di stato e azione venga visitata un numero infinito di volte, e che la politica diventi completamente golosa col tempo. Far decrescere ε progressivamente è il modo abituale per soddisfarla.

Esiste anche il **controllo Monte Carlo**: valuti Q(s,a) facendo la media dei ritorni completi degli episodi e migliori la politica rendendola ε-greedy rispetto a quei valori. Funziona, ma richiede episodi completi e tende ad avere più varianza. SARSA e Q-learning risolvono lo stesso problema aggiornando a ogni passo, grazie al bootstrapping del TD.

> 🎙️ **Traccia della lezione:** collega con la lezione sui bandit: l'ε-greedy era già comparso lì. Chiedi "perché qui serve esplorare, se conosciamo l'ambiente un po' meglio che in un bandit?".


## 3. SARSA: impara la politica che davvero segui

SARSA prende il nome dalla sequenza che usa in ogni aggiornamento: stato, azione, ricompensa, stato successivo, azione successiva. L'agente si trova in uno stato, sceglie un'azione con la sua politica ε-greedy, osserva la ricompensa e il nuovo stato, e poi —il passo chiave— **sceglie anche l'azione successiva con quella stessa politica ε-greedy**, prima di aggiornare. L'aggiornamento usa il Q di quell'azione successiva, quella che verrà davvero eseguita:

```
Q(S,A) si avvicina un po' a: ricompensa ricevuta + valore scontato di Q nello stato successivo,
                             per l'azione che la politica ε-greedy ha scelto davvero
```

Per questo SARSA è **on-policy** (dentro la politica): impara il valore della politica che effettivamente segue, inclusa la propria casualità esplorativa. Se la tua politica ε-greedy a volte commette "errori" esplorando, SARSA incorpora quegli errori nella sua stima di valore, e questo lo rende più prudente: impara a evitare zone in cui un'esplorazione sfortunata potrebbe costare caro.


## 4. Q-learning: impara la politica ottimale, anche se esplori in un altro modo

Q-learning cambia un solo dettaglio, con grandi conseguenze. Invece di usare il Q dell'azione che davvero prenderai dopo, usa il valore **massimo** possibile tra tutte le azioni dello stato successivo, indipendentemente da quella che sceglierai davvero:

```
Q(S,A) si avvicina un po' a: ricompensa ricevuta + valore scontato del miglior Q possibile
                             nello stato successivo (il massimo su tutte le azioni)
```

In questo modo Q-learning diventa **off-policy** (fuori dalla politica): la politica di comportamento (con cui esplori, per esempio ε-greedy) può essere diversa dalla politica obiettivo che stai imparando (quella puramente golosa, l'ottimale). Q-learning impara il valore di "se da ora in poi agissi sempre in modo ottimale", anche mentre continua a esplorare in modo imperfetto. Questa separazione tra "come esploro" e "cosa imparo" permette anche di apprendere da dati raccolti con un'altra politica, o già registrati in anticipo.

> 🎙️ **Traccia della lezione:** scrivi le due formule una sopra l'altra alla lavagna e cerchia l'unica differenza: `Q(S',A')` (l'azione che davvero prenderai) contro `max Q(S',·)` (la migliore azione possibile). È tutto ciò che distingue i due algoritmi.


## 5. L'analogia della scogliera: prudente contro rischioso

Il modo più chiaro per sentire la differenza è il classico esempio della **scogliera** (*Cliff Walking*). È una griglia in cui l'agente parte dall'angolo in basso a sinistra e l'obiettivo si trova nell'angolo in basso a destra. Tra i due, lungo il bordo inferiore, c'è una scogliera: cadere in essa dà una forte penalità e riporta al punto di partenza. Ogni passo normale costa un po', quindi conviene arrivare il prima possibile.

Il percorso più corto costeggia la scogliera, attaccato al bordo. È anche il più rischioso, perché un solo passo falso —un movimento esplorativo casuale dell'ε-greedy— ti fa cadere giù.

- **Q-learning** impara che quel percorso corto e attaccato al bordo è l'ottimale, perché calcola il valore assumendo che da quel momento agirà sempre in modo ottimale, senza considerare che la propria esplorazione potrebbe farlo cadere. Il percorso finale, una volta smesso di esplorare, è il più corto possibile
- **SARSA** impara un percorso più lontano dal bordo, più lungo ma più sicuro, perché la sua stima tiene conto del fatto che, mentre esplora, a volte sbaglia

Durante l'addestramento stesso, mentre entrambi gli agenti continuano a esplorare con ε-greedy, SARSA di solito accumula più ricompensa per episodio perché cade meno volte nella scogliera. Q-learning cade di più durante l'addestramento, ma alla fine impara la politica obiettivo più corta e ottimale. Nessuno dei due "vince" in astratto: ciascuno risponde a una domanda diversa su quale politica stai valutando.

> 🎙️ **Traccia della lezione:** prima di programmare, chiedi alla classe di prevedere ad alta voce quale dei due percorsi disegnerebbe ciascun algoritmo. Conserva la previsione per confrontarla con il grafico dei percorsi alla fine.


## 6. Manos a la obra: Cliff Walking con SARSA e Q-learning

Costruirai l'ambiente della scogliera da zero, senza librerie esterne, e addestrerai un agente con ciascun algoritmo. Ti servono solo `numpy` e `matplotlib`, che sono già preinstallati in Google Colab.

Costruisci il codice cella per cella, eseguendo ciascuna prima di passare alla successiva.


In [ ]:
# Solo necesitas numpy y matplotlib, ya vienen preinstalados en Google Colab
import numpy as np
import matplotlib.pyplot as plt


### L'ambiente: la griglia e la scogliera

Una griglia di quattro righe per 12 colonne (4×12). L'agente parte in basso a sinistra (riga 3, colonna 0) e l'obiettivo si trova in basso a destra (riga 3, colonna 11). Tutta la riga inferiore tra i due, eccetto la partenza e l'obiettivo, è la scogliera.

Le azioni sono quattro: 0=su, 1=giù, 2=sinistra, 3=destra.


In [ ]:
FILAS, COLUMNAS = 4, 12
INICIO = (3, 0)
META = (3, 11)
ACANTILADO = [(3, c) for c in range(1, 11)]
NUM_ACCIONES = 4


### La funzione `paso`: transizione e ricompensa

Dati uno stato e un'azione, `paso` restituisce lo stato successivo, la ricompensa ricevuta e se l'episodio è terminato. Cadere dalla scogliera penalizza molto (−100) e riporta l'agente all'inizio, senza terminare l'episodio. Raggiungere l'obiettivo dà −1 e termina. Qualsiasi altro movimento costa −1, il prezzo di ogni passo.

> 🎙️ **Traccia della lezione:** chiedi "perché cadere dalla scogliera non termina l'episodio?". Risposta: così l'agente continua a imparare dentro lo stesso episodio invece di riavviare il conteggio della ricompensa.


In [ ]:
def paso(estado, accion):
    """Aplica una acción y devuelve (nuevo_estado, recompensa, terminado)."""
    fila, col = estado
    if accion == 0:
        fila = max(fila - 1, 0)
    elif accion == 1:
        fila = min(fila + 1, FILAS - 1)
    elif accion == 2:
        col = max(col - 1, 0)
    elif accion == 3:
        col = min(col + 1, COLUMNAS - 1)

    nuevo_estado = (fila, col)

    if nuevo_estado in ACANTILADO:
        # Caer por el acantilado: penalización fuerte y vuelta al inicio
        return INICIO, -100, False
    if nuevo_estado == META:
        return nuevo_estado, -1, True
    return nuevo_estado, -1, False


### Verifica rapida

Prima di continuare, verifica che `paso` si comporti come previsto.

> 🎙️ **Traccia della lezione:** chiedi a qualcuno di prevedere ad alta voce il risultato di `paso((2, 1), 1)` (muoversi "giù" verso la scogliera) prima di eseguire la cella.


In [ ]:
print("Moverse 'abajo' desde (2, 1), justo encima del acantilado:", paso((2, 1), 1))
print("Moverse 'derecha' desde el inicio (3, 0):", paso(INICIO, 3))
print("Moverse 'arriba' desde el inicio (3, 0):", paso(INICIO, 0))


### La politica ε-greedy

Con probabilità `epsilon` sceglie un'azione a caso (esplora); altrimenti sceglie l'azione con il Q più alto noto (sfrutta). Se ci sono più azioni a parimerito, `np.random.choice` risolve il pareggio a caso invece di tenersi sempre la prima.


In [ ]:
def politica_epsilon_greedy(Q, estado, epsilon):
    """Con probabilidad epsilon explora; si no, elige la mejor acción conocida."""
    if np.random.rand() < epsilon:
        return np.random.randint(NUM_ACCIONES)
    valores = Q[estado]
    return np.random.choice(np.flatnonzero(valores == valores.max()))


### Addestrare con SARSA

Osserva la riga chiave: `siguiente_accion` viene scelta **prima** di aggiornare, con la stessa politica ε-greedy, ed è quel Q che entra nell'obiettivo. È ciò che rende SARSA on-policy.


In [ ]:
def entrenar_sarsa(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        accion = politica_epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        terminado = False

        while not terminado:
            siguiente_estado, recompensa, terminado = paso(estado, accion)
            siguiente_accion = politica_epsilon_greedy(Q, siguiente_estado, epsilon)

            # Actualización SARSA: usa el Q de la acción que SÍ se va a tomar
            objetivo = recompensa + gamma * Q[siguiente_estado][siguiente_accion]
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado, accion = siguiente_estado, siguiente_accion
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


### Addestrare con Q-learning

Confronta questa funzione con la precedente: la struttura del ciclo è quasi identica, ma l'aggiornamento non dipende più dall'azione che sceglierai dopo, bensì dal Q **massimo** dello stato successivo. Questa è l'unica differenza reale tra i due algoritmi.


In [ ]:
def entrenar_q_learning(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        recompensa_total = 0
        terminado = False

        while not terminado:
            accion = politica_epsilon_greedy(Q, estado, epsilon)
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # Actualización Q-learning: usa el MÁXIMO Q del estado siguiente,
            # sin importar qué acción se elija después en la práctica
            mejor_valor_siguiente = Q[siguiente_estado].max()
            objetivo = recompensa + gamma * mejor_valor_siguiente
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado = siguiente_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


### Estrarre il percorso appreso

Una volta addestrato, `extraer_ruta_greedy` segue la politica puramente golosa (senza esplorazione, `epsilon=0`) a partire dall'inizio. È il percorso che l'agente sceglierebbe se smettessi del tutto di esplorare: la politica che ha davvero appreso.


In [ ]:
def extraer_ruta_greedy(Q, max_pasos=50):
    """Sigue la política puramente codiciosa (sin exploración) desde el inicio."""
    estado = INICIO
    ruta = [estado]
    for _ in range(max_pasos):
        if estado == META:
            break
        accion = np.argmax(Q[estado])
        estado, _, terminado = paso(estado, accion)
        ruta.append(estado)
        if terminado:
            break
    return ruta


### Esegui l'addestramento

Fissa il seme casuale perché il risultato sia riproducibile e addestra i due agenti con gli stessi iperparametri: 500 episodi, α=0.5, γ=1.0, ε=0.1.

> 🎙️ **Traccia della lezione:** mentre viene eseguito (è quasi istantaneo), ricorda ad alta voce la frase chiave di oggi: "SARSA impara il valore di ciò che fa davvero; Q-learning impara il valore dell'ottimo".


In [ ]:
np.random.seed(0)
Q_sarsa, recompensas_sarsa = entrenar_sarsa()
Q_qlearning, recompensas_qlearning = entrenar_q_learning()

print("Episodios entrenados:", len(recompensas_sarsa))


### Grafico 1: ricompensa per episodio durante l'addestramento

È lisciato con una media mobile perché, da episodio a episodio, c'è molto rumore: una sola caduta dalla scogliera (−100) destabilizza il numero grezzo.


In [ ]:
def media_movil(valores, ventana=20):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")

plt.figure(figsize=(9, 5))
plt.plot(media_movil(recompensas_sarsa), label="SARSA (on-policy)")
plt.plot(media_movil(recompensas_qlearning), label="Q-learning (off-policy)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa por episodio (media móvil)")
plt.title("Cliff Walking: recompensa durante el entrenamiento")
plt.legend()
plt.tight_layout()
plt.show()


**Come leggere il grafico:** la curva di SARSA di solito sta sopra quella di Q-learning per gran parte dell'addestramento, perché cade meno volte dalla scogliera mentre esplora. Q-learning paga, durante l'addestramento, il prezzo di calcolare il suo obiettivo assumendo una politica ottimale che in pratica non sta ancora seguendo.

> 🎙️ **Traccia della lezione:** chiedi "perché la linea di Q-learning qui è più bassa, se abbiamo detto che impara il percorso ottimale?". Lascia che la classe colleghi la risposta alla differenza on-policy / off-policy prima di proseguire.


### Grafico 2: percorsi finali appresi (politica golosa)

Ora estrai il percorso che ciascun agente seguirebbe senza esplorare, e disegnalo sulla griglia.


In [ ]:
def dibujar_ruta(ax, ruta, titulo):
    cuadricula = np.zeros((FILAS, COLUMNAS))
    for (f, c) in ACANTILADO:
        cuadricula[f, c] = -1
    ax.imshow(cuadricula, cmap="Pastel1")
    filas_ruta = [p[0] for p in ruta]
    columnas_ruta = [p[1] for p in ruta]
    ax.plot(columnas_ruta, filas_ruta, marker="o", color="tab:blue")
    ax.scatter(*INICIO[::-1], color="green", s=100, label="Inicio", zorder=5)
    ax.scatter(*META[::-1], color="red", s=100, label="Meta", zorder=5)
    ax.set_title(titulo)
    ax.invert_yaxis()
    ax.legend(loc="upper center")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
dibujar_ruta(ejes[0], extraer_ruta_greedy(Q_sarsa), "Ruta final de SARSA (más prudente)")
dibujar_ruta(ejes[1], extraer_ruta_greedy(Q_qlearning), "Ruta final de Q-learning (más corta)")
plt.tight_layout()
plt.show()


**Come leggere il grafico:** il percorso di Q-learning di solito si attacca al bordo della scogliera (la riga appena sopra di essa): è il più corto e l'ottimale. Il percorso di SARSA si allontana di una o più righe verso l'alto, sacrificando lunghezza in cambio di sicurezza rispetto alla propria esplorazione. Confronta questo con la previsione fatta dalla classe prima di programmare.

> 🎙️ **Traccia della lezione:** chiedi alla classe di contare quanti passi ha ciascun percorso e di mettere in relazione la differenza con la ricompensa del grafico precedente: più passi, più volte si paga il costo di −1 per movimento.


## 7. Bias di sovrastima e Double Q-learning

Q-learning ha un effetto collaterale sottile: poiché il suo aggiornamento usa il **massimo** tra diverse stime rumorose di Q, quel massimo tende a essere distorto verso l'alto, esattamente come se scegliessi sempre "il candidato migliore" tra diverse misurazioni imprecise: di solito finisci per tenerti una che esagera per caso. Questo è noto come **bias di sovrastima** (*overestimation bias*).

**Double Q-learning** lo corregge mantenendo due tabelle di valori indipendenti: una decide quale azione sembra migliore, e l'altra valuta quanto vale davvero quell'azione (scambiando i ruoli a ogni passo). Separando "chi sceglie" da "chi valuta", si elimina gran parte del bias.

Esiste anche la **SARSA attesa** (*Expected SARSA*): invece di usare il Q dell'azione che si prende davvero (SARSA) o il massimo (Q-learning), usa la media ponderata di Q su tutte le azioni possibili, secondo la probabilità di ciascuna sotto la politica. Quando la politica obiettivo è completamente golosa, questa versione coincide esattamente con Q-learning.


## Punti chiave

- Il controllo cerca di migliorare la politica, non solo di valutarla: alterna valutazione e miglioramento seguendo il principio dell'Iterazione di Politica Generalizzata (GPI)
- La politica ε-greedy bilancia esplorazione e sfruttamento; la condizione GLIE garantisce la convergenza se visiti abbastanza e rendi la politica golosa col tempo
- SARSA è on-policy: impara il valore della politica che effettivamente segue, inclusa la sua esplorazione, il che la rende più prudente
- Q-learning è off-policy: impara il valore della politica ottimale anche se esplora in un altro modo, il che la rende più rischiosa durante l'addestramento ma ottimale alla fine
- L'esempio della scogliera lo mostra con chiarezza: SARSA accumula più ricompensa mentre si addestra, Q-learning trova il percorso più corto
- L'operatore massimo di Q-learning introduce un bias di sovrastima; Double Q-learning lo corregge separando selezione e valutazione in due tabelle


## Chiusura e prossimi passi

Oggi hai trasformato la valutazione della lezione precedente in controllo: due algoritmi, la stessa domanda ("quanto è buona questa azione?"), due risposte diverse a seconda che tu impari il valore di ciò che fai (SARSA) o il valore dell'ottimo (Q-learning).

Finora hai conservato tutti i valori Q in una tabella, una casella per ogni combinazione di stato e azione. Quella tabella funziona perfettamente su una griglia di 12 colonne, ma smette di essere praticabile non appena il numero di stati esplode —per esempio, quando lo stato sono i pixel dello schermo di un videogioco—. Nella Lezione 7 sostituirai quella tabella con una rete neurale che impara ad approssimare Q(s,a): così nascono le reti Q profonde (DQN, dall'inglese *Deep Q-Networks*).

**Prima della prossima lezione:**
- **Apri** il notebook di pratica `clase06_practica_control_sin_modelo.ipynb` e completa i due esercizi
- **Sperimenta** alzando `epsilon` a 0.3 nelle due funzioni di addestramento: osserva se il divario tra la ricompensa di SARSA e quella di Q-learning si allarga o si riduce
- **Cambia** la penalità della scogliera in `paso` (per esempio, da −100 a −10 o a −500) e confronta come cambiano i due percorsi finali

> 🎙️ **Traccia della lezione:** chiudi ripetendo la frase chiave: "ricompensa immediata + valore scontato dello stato successivo, per l'azione che seguo (SARSA) o per la migliore azione possibile (Q-learning)". È la stessa struttura del TD che hanno visto nella lezione precedente, ora applicata a migliorare la politica invece di limitarsi a misurarla.
